GitHub URL: https://github.com/hagsetolav-lgtm/Deliverable_2.git

Imports which are used in the deliverable:

In [1]:
from pathlib import Path
import re

from typing import Any
import json

import numpy as np

import pandas as pd

Task 1: File ingestion and regular expressions.

Part 1 of Task 1: Robust file reading function.

In [2]:
def read_log_file(file_path: Path) -> tuple[str, str]:
    """Read a log file using either Latin-1 or UTF-8 encoding.
    
    Args:
        file_path: Path to the log file.
        
    Returns:
        A tuple containing decoded text and encoding used.
    """

    raw_data = file_path.read_bytes()

    try:
        return raw_data.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw_data.decode("latin-1"), "latin-1"

This function reades the bytes and first tries the utf-8 encoding.
If that fails, it will try the Latin-1 encoding.

Part 2 of Task 1: Log parser function.

In [3]:
def parse_log(file_text: str) -> list[dict[str, str | float]]:
    """Extract the transaction records from log text.
    
    Args:
        file_text: Log text including transactions and noise.
        
    Returns:
        A list of dictionaries with timestamp, register_id, item,
        price, currency and status. Prices as floats, other values
        as strings.
    """
    pattern = re.compile(
        r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+"
        r"\[(?P<register_id>POS_\d+)\]\s+"
        r"item=(?P<item>.+?)\s+"
        r"price=(?P<price>-?\d+(?:\.\d+)?)\s+"
        r"(?P<currency>[A-Z]{3})\s+"
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
    )

    transactions: list[dict[str, str | float]] = []

    for line in file_text.splitlines():
        match = pattern.fullmatch(line.strip())

        if match is not None:
            record: dict[str, str | float] = {
                "timestamp": match.group("timestamp"),
                "register_id": match.group("register_id"),
                "item": match.group("item"),
                "price": float(match.group("price")),
                "currency": match.group("currency"),
                "status": match.group("status"),
            }
            transactions.append(record)

    return transactions

This function extracts six fields from each valid transaction line.
It will ignore lines which do not match the expected format. It will
return the transactions as a list of dictionaries.

Task 2: Recursive utility functions.

Part 1 of Task 2: Recursive directory search function.

In [4]:
def find_log_files(directory: str | Path) -> list[Path]:
    """Find all .log files inside the directory.
    
    Args: 
        directory: the path to the directory to search.
        
    Returns:
        A list of Path objects for all .log files that are found, 
        including files in nested subdirectories.
    """
    directory = Path(directory)
    log_files: list[Path] = []

    for item in directory.iterdir():
        if item.is_dir():
            log_files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

This function finds all the .log files which are inside the directory, examines 
the file paths and adds them in the subfolder to the list (.extend()) and adds an 
inidvidual .log file (.append()).

Part 2 of Task 2: Recursive function.

In [5]:
def find_active_registers(data: Any) -> set[str]:
    """Collect the active register IDs from the json file.
    
    Args:
        data: A decoded JSON value, with dictionaries, lists and individual values.

    Returns:
        A set of register IDs whose active value is True.
    """

    active_ids: set[str] = set()

    if isinstance(data, dict):
        register_id = data.get("id")

        if data.get("active") is True and isinstance(register_id, str):
            active_ids.add(register_id)

        for value in data.values():
            active_ids.update(find_active_registers(value))

    elif isinstance(data, list):
        for item in data:
            active_ids.update(find_active_registers(item))

    return active_ids

The function checks for active registers in dictionaries, search every item in the lists
and returns IDs it finds.

In [6]:
with Path("data/store_config.json").open(encoding="utf-8") as file:
    store_config = json.load(file)

active_registers = find_active_registers(store_config)
print("Active register IDs:", sorted(active_registers))

Active register IDs: ['POS_01', 'POS_02', 'POS_03']


Loads the JSON into dictionaries and lists and finds the active register IDs.

Task 3: Functions for cleaning and analysing data using NumPy.

Part 1 of Task 3: Function for removing price outliers.

In [7]:
def clean_price_outliers(prices: list[float] | np.ndarray) -> np.ndarray:
    """Replaces price outliers with median of non-outlier prices.
    
    Args: 
        prices: Numeric price data converts to 1D float array.
        
    Returns:
        Clean 1D float array. Values more than three standard deviations 
        from the mean are replaced by the median of the remaining prices.
    """

    cleaned = np.array(prices, dtype=float).flatten()

    if cleaned.size == 0:
        return cleaned

    mean_price = np.mean(cleaned)
    std_price = np.std(cleaned)

    outliers = np.abs(cleaned - mean_price) > 3 * std_price

    if np.any(outliers):
        valid_prices = cleaned[~outliers]
        cleaned[outliers] = np.median(valid_prices)

    return cleaned


Converts the input into a 1D float array, identifies outliers and replaces them with median of non-outliers.

Part 2 of Task 3: Function for refund ratio.

In [8]:
def compute_refund_ratio(
        sales_count: np.ndarray,
        refund_count: np.ndarray,
) -> np.ndarray:
    """Caclulate the refund ratio for each pair of counts.
    
    Args:
        sales_count: Array of sales counts for each item.
        refund_count: Array of refund counts in the same order.

    Returns:
        An array of refund ratios. A small value of 1e-6 is added
        to each sales count to prevent division by zero.
    """
    return refund_count / (sales_count + 1e-6)

Calculates refund ratio and returns an array.

Task 4: Data analysis using Pandas.

Part 1 Task 4: Conversion of transaction dictionaries to DataFrame.

In [9]:
def create_transaction_dataframe(
        transactions: list[dict[str, str | float]],
) -> pd.DataFrame:
    """Convert transaction records into DataFrame

    Args: 
    transactions: Transaction dictionaries returned by log parser.

    Returns:
    A DataFrame with datetime index sorted from earliest to latest.
    """
    columns = [
        "timestamp", "register_id", "item", 
        "price", "currency", "status",
    ]
    df = pd.DataFrame(transactions, columns=columns)

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format= "%Y-%m-%d %H:%M:%S",
    )

    df = df.set_index("timestamp")
    df = df.sort_index()

    return df


Creates a table from dictionaries, converts timestamp strings into datetime values.
Uses timestamp as the row index, and places the earliest transactions first.

Part 2 Task 4:

In [10]:
def compute_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the price per item in one-hour windows.
    
    Args:
        df: Transaction DataFrame with a DatetimeIndex and columns
        named item and price.

    Returns:
        A DataFrame containing total price for each item and hour,
        indexed by item and timestamp.
    """
    hourly_revenue = (
        df.groupby("item")["price"]
        .resample("1h")
        .sum()
        .to_frame(name="total_price")
    )

    return hourly_revenue

The function separates transactions by product into hourly windows, and adds the prices, finally returning result as DataFrame.

Part 3 Task 4: Table for total transaction revenue.

In [11]:
def create_status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize total transaction prices by item and status.
    
    Args: 
        df: Transaction DataFrame containing item, status and price.

    Returns:
        A pivot table with items as rows and SUCCESS, REFUND, and
        CANCELLED as columns. Missing combinations contain zero.
    """
    pivot = df.pivot_table(
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0,
    )

    return pivot.reindex(
        columns=["SUCCESS", "REFUND", "CANCELLED"],
        fill_value=0,
    )

Function for creating the pivot table. aggfunc=0 adds prices for each item/status combination, reindex() makes sure that all columns appear. fill_value=0 fills missing combiations.

Part 4 Task 4: DataFrame as input, calculates  the refund ratio per item.

In [12]:
def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the refund ratio for each item using the transaction counts.
    
    Args:
        df: Transaction DataFrame containing item and status columns.
        
    Returns: 
        DataFrame indexed by item with sales_count, refund_count, and 
        refund_ratio columns. Sales counts include SUCCESS transactions;
        refund counts include REFUND transactions
    """
    counts =(
        df.groupby(["item", "status"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["SUCCESS", "REFUND"], fill_value=0)
    )

    result = counts.rename(columns={
        "SUCCESS": "sales_count",
        "REFUND": "refund_count",
    })
    result.columns.name = None

    result["refund_ratio"] = compute_refund_ratio(
        result["sales_count"].to_numpy(),
        result["refund_count"].to_numpy(),
    )

    return result

The function calculates the ratio for each item. Returns a DataFrame.

Task 5: Create full workflow.

In [13]:
data_directory = Path("data")

log_files = sorted(find_log_files(data_directory))

with (data_directory / "store_config.json").open(encoding="utf-8") as file:
    store_config = json.load(file)

active_registers = find_active_registers(store_config)
print(f"Found {len(log_files)} log files.")
print("Active register IDs:", sorted(active_registers))

Found 9 log files.
Active register IDs: ['POS_01', 'POS_02', 'POS_03']


This finds log files, reads the store configuration and extracts the active register IDs using the function from task 2.

In [14]:
transactions: list[dict[str, str | float]] = []

for log_file in log_files:
    file_text, encoding = read_log_file(log_file)
    records = parse_log(file_text)
    transactions.extend(records)

    print(f"{log_file.name}: {len(records)} records ({encoding})")

print(f"\nTotal transactions: {len(transactions)}")

register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)
register_sales.log: 80 records (utf-8)

Total transactions: 720


Collects the transaction records from all log files and combines them into a list with extend().

In [15]:
df = create_transaction_dataframe(transactions)

print(f"DataFrame contains {len(df)} transactions.")
print(f"Datetime index:", isinstance(df.index, pd.DatetimeIndex))
print("Chronologically sorted:", df.index.is_monotonic_increasing)

DataFrame contains 720 transactions.
Datetime index: True
Chronologically sorted: True


Converts transactions into DataFrame, sorted by timestamp.

In [16]:
cleaned_df = df.copy()
original_prices = df["price"].to_numpy()
cleaned_prices = clean_price_outliers(original_prices)

cleaned_df["price"] = cleaned_prices

print("Prices replaced:", np.count_nonzero(original_prices != cleaned_prices))
print("Remaining 999.00 prices:", (cleaned_df["price"] == 999.0).sum())


Prices replaced: 23
Remaining 999.00 prices: 0


Replaces outliers with the median of the valid prices.

In [17]:
corrected_df = cleaned_df.copy()
nok_mask = corrected_df["currency"] == "NOK"
corrected_df.loc[nok_mask, "price"] *= 10.0

print("NOK prices corrected:", nok_mask.sum())

NOK prices corrected: 246


Multiplies NOK prices by 10.

In [18]:
hourly_revenue = compute_hourly_revenue(corrected_df)
status_pivot = create_status_pivot(corrected_df)
item_refund_ratios = calculate_item_refund_ratios(corrected_df)

print("Hourly revenue:")
display(hourly_revenue.head())

print("Revenue by item and status:")
display(status_pivot)

print("Refund ratios by item:")
display(item_refund_ratios)

Hourly revenue:


total_price
item       timestamp                       
Cappuccino 2026-10-15 07:00:00      1082.66
           2026-10-15 08:00:00       750.96
           2026-10-15 09:00:00       169.70
Croissant  2026-10-15 07:00:00       935.47
           2026-10-15 08:00:00       779.76

Revenue by item and status:


status,SUCCESS,REFUND,CANCELLED
item,,,
Cappuccino,1421.58,296.02,285.72
Croissant,1424.55,185.07,271.94
Espresso,993.57,245.48,259.69
Latte,1266.08,431.50,278.33
Muffin,1155.28,280.40,338.91
Sandwich,2817.53,452.39,637.66


Refund ratios by item:


,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,76,15,0.197368
Croissant,86,20,0.232558
Espresso,82,26,0.317073
Latte,75,25,0.333333
Muffin,78,17,0.217949
Sandwich,78,17,0.217949


Generating reports from the cleaned and currency-corrected data.

In [19]:
corrected_df.to_csv("cleaned_transactions.csv", index=True)

hourly_revenue.reset_index().to_json(
    "hourly_revenue.json",
    orient="records",
    date_format="iso",
    indent=4,
)

status_pivot.to_csv("status_pivot_table.csv", index=True)
item_refund_ratios.to_csv("item_refund_ratios.csv", index=True)

print("All four output files saved.")

All four output files saved.


Saving the four output files.